# Stored execution evidence — Aerial Cactus Identification (inference)

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `3d00c21a42c897cce23d49199ca4248d6c2bfdf8eec9f2d1ab9e2885e9adb4f0`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Agent — Aerial Cactus Kaggle inference

This notebook is the Kaggle-side submission notebook. Before running it, attach both inputs:

1. Competition data: `aerial-cactus-identification`
2. The private Kaggle Dataset created from `aerial_cactus_dinov3_assets.zip`

Select a GPU accelerator, keep Internet off, and use **Save Version → Save & Run All**. The final code cell creates `/kaggle/working/submission.csv` for the Kernels-only competition submission.


In [1]:
# Locate the attached competition data and packaged Jiaozi checkpoint

import json
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import AutoConfig, AutoModel

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook must run in Kaggle with attached inputs.")

sample_candidates = sorted(KAGGLE_INPUT.rglob("sample_submission.csv"))
sample_candidates = [
    path
    for path in sample_candidates
    if "aerial-cactus-identification" in str(path.parent).lower()
]
if not sample_candidates:
    raise FileNotFoundError(
        "Competition sample_submission.csv not found. Add the "
        "aerial-cactus-identification competition as an input."
    )

SAMPLE_PATH = sample_candidates[0]
COMPETITION_ROOT = next(
    parent
    for parent in [SAMPLE_PATH.parent, *SAMPLE_PATH.parents]
    if parent.name == "aerial-cactus-identification"
)

# Kaggle may expose uploaded zip contents directly or retain the zip.
checkpoint_candidates = sorted(KAGGLE_INPUT.rglob("best_model.pt"))

if not checkpoint_candidates:
    asset_zips = sorted(
        path
        for path in KAGGLE_INPUT.rglob("*.zip")
        if "aerial_cactus_dinov3_assets" in path.name.lower()
    )
    if not asset_zips:
        raise FileNotFoundError(
            "best_model.pt or aerial_cactus_dinov3_assets.zip was not found. "
            "Attach the private asset Dataset to this notebook."
        )

    extracted_assets = WORKING_DIR / "restored_assets"
    extracted_assets.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(asset_zips[0], "r") as archive:
        archive.extractall(extracted_assets)
    checkpoint_candidates = sorted(extracted_assets.rglob("best_model.pt"))

CHECKPOINT_PATH = checkpoint_candidates[0]
ASSET_DIR = CHECKPOINT_PATH.parent
CONFIG_PATH = ASSET_DIR / "config.json"

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"DINOv3 config is missing: {CONFIG_PATH}")

sample = pd.read_csv(SAMPLE_PATH)
if sample.columns.tolist() != ["id", "has_cactus"]:
    raise ValueError(f"Unexpected sample columns: {sample.columns.tolist()}")

print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("competition root:", COMPETITION_ROOT)
print("sample submission:", SAMPLE_PATH)
print("checkpoint:", CHECKPOINT_PATH)
print("test rows:", len(sample))


torch: 2.10.0+cu128
GPU available: True
competition root: /kaggle/input/competitions/aerial-cactus-identification
sample submission: /kaggle/input/competitions/aerial-cactus-identification/sample_submission.csv
checkpoint: /kaggle/input/datasets/baron66/jiaozi-aerial-cactus-dinov3-assets/best_model.pt
test rows: 4000


In [2]:
# Restore all DINOv3 tensors and the trained binary classification head

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    raw_state = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    raw_state = checkpoint["state_dict"]
elif isinstance(checkpoint, dict):
    raw_state = checkpoint
else:
    raise TypeError(f"Unsupported checkpoint type: {type(checkpoint)}")

raw_state = {
    str(key).removeprefix("module."): value
    for key, value in raw_state.items()
    if torch.is_tensor(value)
}

config = AutoConfig.from_pretrained(ASSET_DIR, local_files_only=True)
backbone = AutoModel.from_config(config)
reference_state = backbone.state_dict()

mapped_backbone_state = {}
source_keys_used = set()

for target_key, target_tensor in reference_state.items():
    candidates = [
        (source_key, source_tensor)
        for source_key, source_tensor in raw_state.items()
        if (
            source_key == target_key
            or source_key.endswith("." + target_key)
        )
        and tuple(source_tensor.shape) == tuple(target_tensor.shape)
    ]
    if candidates:
        source_key, source_tensor = min(candidates, key=lambda item: len(item[0]))
        mapped_backbone_state[target_key] = source_tensor
        source_keys_used.add(source_key)

match_ratio = len(mapped_backbone_state) / max(1, len(reference_state))
print(
    "backbone tensors restored:",
    f"{len(mapped_backbone_state)}/{len(reference_state)} ({match_ratio:.1%})",
)

if match_ratio < 0.95:
    raise RuntimeError(
        "Checkpoint-to-DINOv3 mapping is incomplete; refusing unsafe inference."
    )

backbone.load_state_dict(mapped_backbone_state, strict=False)

head_candidates = [
    (key, tensor)
    for key, tensor in raw_state.items()
    if key not in source_keys_used
    and tensor.ndim == 2
    and tensor.shape[0] == 2
    and tensor.shape[1] >= 128
]
if not head_candidates:
    raise RuntimeError("Binary classification head was not found in the checkpoint.")

head_weight_key, head_weight = min(
    head_candidates,
    key=lambda item: item[1].shape[1],
)
head_prefix = head_weight_key.rsplit(".", 1)[0]
head_bias = raw_state.get(head_prefix + ".bias")
if head_bias is None or tuple(head_bias.shape) != (2,):
    raise RuntimeError("Matching classification-head bias was not found.")

head = nn.Linear(head_weight.shape[1], 2)
with torch.no_grad():
    head.weight.copy_(head_weight)
    head.bias.copy_(head_bias)


class RestoredCactusModel(nn.Module):
    def __init__(self, backbone_model, classifier):
        super().__init__()
        self.backbone = backbone_model
        self.classifier = classifier

    def forward(self, pixel_values):
        output = self.backbone(pixel_values=pixel_values)
        pooled = getattr(output, "pooler_output", None)
        if pooled is None:
            pooled = output.last_hidden_state[:, 0]
        return self.classifier(pooled)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RestoredCactusModel(backbone, head).to(device).eval()

print("classification head:", head_weight_key, tuple(head_weight.shape))
print("best epoch:", checkpoint.get("best_epoch"))
print("best validation ROC-AUC:", checkpoint.get("best_metric"))
print("device:", device)


backbone tensors restored: 211/211 (100.0%)
classification head: head.classifier.weight (2, 768)
best epoch: 10
best validation ROC-AUC: 0.9998905541231663
device: cuda


In [3]:
# Run Kaggle test inference and write /kaggle/working/submission.csv

test_ids = sample["id"].astype(str).tolist()

# First check whether the test images have already been extracted
first_matches = [
    path
    for path in COMPETITION_ROOT.rglob(test_ids[0])
    if path.is_file()
]

# Older competitions often store test images in test.zip; extract it when necessary
if not first_matches:
    test_zip_candidates = sorted(COMPETITION_ROOT.rglob("test.zip"))

    if not test_zip_candidates:
        test_zip_candidates = sorted(
            path
            for path in COMPETITION_ROOT.rglob("*.zip")
            if "test" in path.name.lower()
        )

    if not test_zip_candidates:
        raise FileNotFoundError(
            f"Neither an extracted test image nor test.zip was found "
            f"under {COMPETITION_ROOT}"
        )

    extracted_test_root = (
        WORKING_DIR / "aerial_cactus_test_images"
    )
    extracted_test_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Extracting Kaggle test archive:",
        test_zip_candidates[0],
    )

    with zipfile.ZipFile(
        test_zip_candidates[0],
        "r",
    ) as archive:
        archive.extractall(extracted_test_root)

    first_matches = [
        path
        for path in extracted_test_root.rglob(test_ids[0])
        if path.is_file()
    ]

if not first_matches:
    raise FileNotFoundError(
        f"First test image not found after extraction: "
        f"{test_ids[0]}"
    )

candidate_dirs = sorted({
    path.parent.resolve()
    for path in first_matches
})

candidate_dirs = sorted({path.parent.resolve() for path in first_matches})
test_image_dir = max(
    candidate_dirs,
    key=lambda directory: sum(
        (directory / image_id).is_file()
        for image_id in test_ids[:100]
    ),
)

missing = [
    image_id
    for image_id in test_ids
    if not (test_image_dir / image_id).is_file()
]
if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} test images; first: {missing[:10]}"
    )

test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class CactusTestDataset(Dataset):
    def __init__(self, ids, image_dir):
        self.ids = ids
        self.image_dir = image_dir

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        with Image.open(self.image_dir / image_id) as image:
            pixel_values = test_transform(image.convert("RGB"))
        return image_id, pixel_values


loader = DataLoader(
    CactusTestDataset(test_ids, test_image_dir),
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

prediction_ids = []
prediction_values = []

with torch.inference_mode():
    for batch_index, (batch_ids, pixel_values) in enumerate(loader, start=1):
        pixel_values = pixel_values.to(device, non_blocking=True)
        with torch.autocast(
            device_type=device.type,
            enabled=device.type == "cuda",
        ):
            logits = model(pixel_values)
            probabilities = torch.softmax(logits, dim=1)[:, 1]

        prediction_ids.extend(list(batch_ids))
        prediction_values.extend(probabilities.float().cpu().tolist())

        if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(loader):
            print(f"[infer] batch {batch_index}/{len(loader)}")

predictions = pd.DataFrame(
    {"id": prediction_ids, "has_cactus": prediction_values}
)
submission = sample[["id"]].merge(predictions, on="id", how="left")
values = submission["has_cactus"].to_numpy(dtype=float)

assert submission.columns.tolist() == ["id", "has_cactus"]
assert len(submission) == len(sample)
assert submission["id"].astype(str).equals(sample["id"].astype(str))
assert np.isfinite(values).all()
assert ((values >= 0.0) & (values <= 1.0)).all()
assert np.std(values) > 0.0

SUBMISSION_PATH = WORKING_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("\n=== KAGGLE SUBMISSION READY ===")
print("file:", SUBMISSION_PATH)
print("shape:", submission.shape)
print("probability range:", float(values.min()), "to", float(values.max()))
print("probability std:", float(values.std()))
display(submission.head())


Extracting Kaggle test archive: /kaggle/input/competitions/aerial-cactus-identification/test.zip
[infer] batch 1/63
[infer] batch 10/63
[infer] batch 20/63
[infer] batch 30/63
[infer] batch 40/63
[infer] batch 50/63
[infer] batch 60/63
[infer] batch 63/63

=== KAGGLE SUBMISSION READY ===
file: /kaggle/working/submission.csv
shape: (4000, 2)
probability range: 7.68579297427685e-11 to 1.0
probability std: 0.43233570756133344


,id,has_cactus
0,000940378805c44108d287872b2f04ce.jpg,1.000000e+00
1,0017242f54ececa4512b4d7937d1e21e.jpg,1.000000e+00
2,001ee6d8564003107853118ab87df407.jpg,4.793755e-07
3,002e175c3c1e060769475f52182583d0.jpg,7.750694e-07
4,0036e44a7e8f7218e9bc7bf8137e4943.jpg,9.999611e-01
